<h1 style="bg-color:white;color:black; text-align:center">Internee Performance predictor</h1>


Build a machine-learning model to predict intern performance based on task completion rates and feedback.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_percentage_error,mean_absolute_error,root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

Readin the data

In [2]:
df = pd.read_csv("/content/intern_dataset_realistic (1).csv")

checking the dataframe

In [3]:
df.head(3)

,Intern_ID,Completion_Time,Feedback_Rating,Attendance,Performance_Score
0,INT0001,3.26,4.09,86.50,70.98
1,INT0002,NaN,1.33,59.23,42.16
2,INT0003,5.78,NaN,42.98,49.89


## EDA

Data Understanding

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Intern_ID          10000 non-null  object 
 1   Completion_Time    9604 non-null   float64
 2   Feedback_Rating    9608 non-null   float64
 3   Attendance         9610 non-null   float64
 4   Performance_Score  10000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 390.8+ KB


total 5 columns and 4 of which are float only Intern_ID is obj
> - the feedback should be int becuase we want feedback as 1 t 5 rating
> - Drop the Intern_ID Becuase It doesn't provide any information about the target
> - I see some null values in each columns except the perfomance_score and Intern_ID


Descriptive Summary of the Data

In [5]:
df.describe()

,Completion_Time,Feedback_Rating,Attendance,Performance_Score
count,9604.000000,9608.000000,9610.000000,10000.000000
mean,3.958567,2.909313,62.769768,67.808159
std,1.738279,1.312825,26.614348,15.170590
min,1.000000,1.000000,0.000000,25.950000
25%,2.460000,1.800000,52.692500,56.947500
50%,3.920000,2.870000,66.830000,67.830000
75%,5.450000,4.000000,82.857500,78.870000
max,7.000000,5.000000,100.000000,100.000000


we will consider

Completion_time as hours becuase in days it doesn't make any sense
- for the missing values I think I shall replace them with mean

Feedback_rating for this column I have different thing in my head intead of float we will convert it into int becuase usually rating is integer not float using floor round function and
- null values will be replaced
with median value

for attendance we will replace the null values with mean

Checking the columns names

In [6]:
df.columns

Index(['Intern_ID', 'Completion_Time', 'Feedback_Rating', 'Attendance',
       'Performance_Score'],
      dtype='object')

Dropping the Intern_ID

In [7]:
# first lets drop internee columns

df.drop(columns=["Intern_ID"], inplace=True)

**Checking the null values percentage**

In [8]:
# first we have check the percentage of null values

for col in df.columns[:3]:
  print(f"Null values percentage {col}: {round(df[col].isnull().sum() / len(df) * 100,2)}")

Null values percentage Completion_Time: 3.96
Null values percentage Feedback_Rating: 3.92
Null values percentage Attendance: 3.9


We can remove the data with Null values but in this task I will fill the null values

converting the Feedback Rating to Int

In [9]:
# lets convert the feedback rating to int
# the issue as the we can't do it with as type becuase of the null values but lets go with round function


df['Feedback_Rating']=round(df['Feedback_Rating'])

checking the data

In [10]:
df

,Completion_Time,Feedback_Rating,Attendance,Performance_Score
0,3.26,4.0,86.50,70.98
1,NaN,1.0,59.23,42.16
2,5.78,NaN,42.98,49.89
3,4.28,1.0,83.16,68.49
4,1.53,2.0,74.10,77.51
...,...,...,...,...
9995,5.86,4.0,97.61,81.17
9996,6.30,2.0,87.39,43.81
9997,6.71,1.0,50.97,36.69
9998,3.38,4.0,70.05,66.16


In [11]:
df['Feedback_Rating'].median()

3.0

Fill null values Function

In [12]:
def fix_null(df):
  for col in df.columns:
    if col != "Feedback_Rating":
      df[col].fillna(df[col].mean(),inplace=True)
    else:
      df[col].fillna(df[col].median(),inplace=True)


Filling the null values

In [13]:
fix_null(df)

/tmp/ipykernel_2151/2935579814.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(),inplace=True)
/tmp/ipykernel_2151/2935579814.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using

checking if the null values are replaced

In [14]:
df.isnull().sum()

,0
Completion_Time,0
Feedback_Rating,0
Attendance,0
Performance_Score,0


## Modeling Phase

Splitting data into X (predicting features) and y (target)  and applying the Train test Split

In [15]:

# the train test split
X = df.drop(columns='Performance_Score')
y = df['Performance_Score']
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42, test_size=0.8)


**Training the Random forest Regressor model**

In [16]:
# now lets train our model

model = RandomForestRegressor(n_estimators=200,max_depth=15)
model.fit(X_train,y_train)

RandomForestRegressor(max_depth=15, n_estimators=200)

Predicting for X_test

In [17]:
ypred = model.predict(X_test)

**Random Forest Evaluation**

In [18]:
#now lets evaluate

print("MSE : ",round(mean_squared_error(y_true=y_test,y_pred=ypred),2))
print("RMSE : ",round(root_mean_squared_error(y_test,ypred),2))
print("MAE : ",round(mean_absolute_error(y_test,ypred),2))

MSE :  112.1
RMSE :  10.59
MAE :  8.55


The above metrices tell us how much our prdiction might be off by
The MAE score tell us how much our prediction is off by which is 8.55 it is alot but since our data have only 3 predicting features it is relatively fine but the model can be improved

In [19]:
# lets tune this model now
estimators = [100,250,300,500]
depts = [5,20,30,40]

for est in estimators:
  for dpt in depts:
    model = RandomForestRegressor(n_estimators=est,max_depth=dpt)
    model.fit(X_train,y_train)
    ypred = model.predict(X_test)

    print(f"MSE Estimators {est} dept {dpt} : ",round(mean_squared_error(y_true=y_test,y_pred=ypred),2))
    print(f"RMSE Estimators {est} dept {dpt} : ",round(root_mean_squared_error(y_test,ypred),2))
    print(f"MAE Estimators {est} dept {dpt} : ",round(mean_absolute_error(y_test,ypred),2))
    print("--"*20)

MSE Estimators 100 dept 5 :  100.22
RMSE Estimators 100 dept 5 :  10.01
MAE Estimators 100 dept 5 :  8.17
----------------------------------------
MSE Estimators 100 dept 20 :  112.98
RMSE Estimators 100 dept 20 :  10.63
MAE Estimators 100 dept 20 :  8.6
----------------------------------------
MSE Estimators 100 dept 30 :  112.67
RMSE Estimators 100 dept 30 :  10.61
MAE Estimators 100 dept 30 :  8.57
----------------------------------------
MSE Estimators 100 dept 40 :  112.7
RMSE Estimators 100 dept 40 :  10.62
MAE Estimators 100 dept 40 :  8.58
----------------------------------------
MSE Estimators 250 dept 5 :  100.24
RMSE Estimators 250 dept 5 :  10.01
MAE Estimators 250 dept 5 :  8.17
----------------------------------------
MSE Estimators 250 dept 20 :  112.49
RMSE Estimators 250 dept 20 :  10.61
MAE Estimators 250 dept 20 :  8.56
----------------------------------------
MSE Estimators 250 dept 30 :  112.84
RMSE Estimators 250 dept 30 :  10.62
MAE Estimators 250 dept 30 :  8.58

from our tuning of hyper parameters we can see that the best parameters for our Random Forest Regression model is
- Estimator 250 and Dept 5 and
- Estimator 300 and depth of 5
- Estimators 500 and dept 5

___

**Modeling with XGBoost Regressor**

In [20]:
#now lets train some other models like linear regression and KNN regressor and xgboost

from xgboost import XGBRegressor

model = XGBRegressor( n_estimators=80,
                      max_depth=3,
                      learning_rate=0.1,
                      random_state=42)
model.fit(X_train,y_train)
ypred = model.predict(X_test)

print(f"MSE Estimators : ",round(mean_squared_error(y_true=y_test,y_pred=ypred),2))
print(f"RMSE Estimators : ",round(root_mean_squared_error(y_test,ypred),2))
print(f"MAE Estimators : ",round(mean_absolute_error(y_test,ypred),2))


MSE Estimators :  96.75
RMSE Estimators :  9.84
MAE Estimators :  8.02


The error is improved from Random forest regressor we can that our model error is now off 8 it is improved from best model of random forest which was giving 8.16 MAE and MSE of 100

___

**Modeling KNN Regressor**

In [21]:
# now lets do the KNN Regressor
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=15)

model.fit(X_train,y_train)
ypred = model.predict(X_test)

print(f"MSE Estimators : ",round(mean_squared_error(y_true=y_test,y_pred=ypred),2))
print(f"RMSE Estimators : ",round(root_mean_squared_error(y_test,ypred),2))
print(f"MAE Estimators : ",round(mean_absolute_error(y_test,ypred),2))


MSE Estimators :  109.42
RMSE Estimators :  10.46
MAE Estimators :  8.57


**Modeling with Linear Regression**

In [22]:
# lets do it with linear regression
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train,y_train)
ypred = model.predict(X_test)

print(f"MSE Estimators : ",round(mean_squared_error(y_true=y_test,y_pred=ypred),2))
print(f"RMSE Estimators : ",round(root_mean_squared_error(y_test,ypred),2))
print(f"MAE Estimators : ",round(mean_absolute_error(y_test,ypred),2))

MSE Estimators :  100.92
RMSE Estimators :  10.05
MAE Estimators :  8.17


<h1 style="bg-color:white;color:black; text-align:center">End of Task 1</h1>